In [ ]:
import os

from pynq import Overlay
from pynq import 

import pprint

In [ ]:
SYSTOLIC_ARRAY_3 = 16

## Load and Write bitstream

In [ ]:
pp = pprint.PrettyPrinter(indent=4)

bitstream_name = "design_1_wrapper"
bitstream_path = f"./bitstream/{bitstream_name}.bit"

if not os.path.exists(bitstream_path):
    raise Exception(f"Could not find {bitstream_path}")    

print(f"Loading {bitstream_name}.bit and parsing .hwh...\n")

# This single line flashes the FPGA AND parses the .hwh file
overlay = Overlay(bitstream_path)

In [ ]:
print("--- IP Dictionary (AXI Memory Map) ---")
# Lists all AXI IPs and their base addresses.
pp.pprint(overlay.ip_dict)

print()
print("--- Clock Configuration ---")
# Shows frequencies of PL clocks driven by Zynq.
pp.pprint(overlay.clock_dict)

print()
print("--- Interrupt Pins ---")
# Lists hardware interrupts to Zynq IRQ controller.
pp.pprint(overlay.interrupt_pins)

print()
print("--- GPIO Dictionary ---")
# Lists AXI GPIO blocks.
pp.pprint(overlay.gpio_dict)

## Write Test Data

# Access the DMA block
dma = overlay.axi_dma_0

# 2. Define the payload size
# 1024 elements of 32-bit integers = 4 Kilobytes of data
DATA_LENGTH = 1024 

# 3. Allocate Contiguous Memory Buffers
print(f"Allocating contiguous buffers for {DATA_LENGTH} elements...")
in_buffer = allocate(shape=(DATA_LENGTH,), dtype=np.uint32)
out_buffer = allocate(shape=(DATA_LENGTH,), dtype=np.uint32)

# 4. Generate Test Data
# Using a sequential array (0, 1, 2, 3...) instead of random numbers. 
# This makes it incredibly easy to spot dropped or duplicated words in hardware.
generated_array = np.arange(DATA_LENGTH, dtype=np.uint32)

# Copy the generated data into the hardware-accessible buffer
in_buffer[:] = generated_array

# 5. Execute the Transfer
print("Initiating DMA transfer...")

# Best Practice: Always prime the receive channel BEFORE sending data. 
# If the FIFO sends data back before the recvchannel is ready to catch it, the AXI stream can lock up.
dma.recvchannel.transfer(out_buffer)
dma.sendchannel.transfer(in_buffer)

# Block until the hardware asserts the TLAST signal indicating completion
dma.sendchannel.wait()
dma.recvchannel.wait()

print("Transfer complete! Verifying data...\n")

# 6. Verify and Debug
if np.array_equal(in_buffer, out_buffer):
    print("✅ SUCCESS: Received data perfectly matches the generated array!")
else:
    print("❌ FAILURE: Data mismatch detected.")
    
    # Find exactly where the arrays differ to help debug AXI-Stream issues
    mismatch_indices = np.where(in_buffer != out_buffer)[0]
    print(f"Total corrupted/missing elements: {len(mismatch_indices)}")
    
    # Print the first 5 mismatches so you can see if data is shifted or zeroed out
    print("\nFirst few mismatches:")
    for i in mismatch_indices[:5]:
        print(f"Index [{i}] -> Sent: {in_buffer[i]} | Received: {out_buffer[i]}")

# 7. Clean up
del in_buffer
del out_buffer